In [1]:
%pip -q install -U "transformers>=4.49.0" "accelerate>=0.33.0" pillow tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"  # 或 "Qwen/Qwen3-VL-8B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
)
model.eval()

print("Loaded:", MODEL_ID, "| device:", device, "| dtype:", dtype)


C:\Users\67600\miniconda3\envs\pairgpu\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
C:\Users\67600\miniconda3\envs\pairgpu\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loaded: Qwen/Qwen3-VL-4B-Instruct | device: cuda | dtype: torch.bfloat16


In [2]:
import pandas as pd

CLASSES_TSV = r"D:\O.D.I.L\classes.tsv"  # 改成你的路径
classes_df = pd.read_csv(CLASSES_TSV, sep="\t")

print("Loaded classes_df. Columns =", list(classes_df.columns))
display(classes_df.head(5))


Loaded classes_df. Columns = ['Index', 'timel_id', 'timel_label']


,Index,timel_id,timel_label
0,1,tm-22v2fsax,Joseph interprétant le songe du panetier
1,2,tm-23h9wqfr,hérisson
2,3,tm-25fjujvw,château
3,4,tm-2645zjsy,moustique
4,5,tm-26uybhrd,juin


In [3]:
import pandas as pd

# 1) 防呆：检查 classes_df 是否存在
if "classes_df" not in globals():
    raise NameError("classes_df 尚未定义。请先运行读取 classes.tsv 的 cell。")

# 2) 防呆：检查必需列
need_cols = {"Index", "timel_label"}
missing = need_cols - set(classes_df.columns)
if missing:
    raise ValueError(f"classes_df 缺少列 {missing}；当前列: {list(classes_df.columns)}")

# 3) 清理 + 去空 + 去重（按 Index 去重）
tmp = classes_df.loc[:, ["Index", "timel_label"]].copy()
tmp["Index"] = tmp["Index"].astype(str).str.strip()
tmp["timel_label"] = tmp["timel_label"].astype(str).str.strip()
tmp = tmp[tmp["Index"].ne("") & tmp["timel_label"].ne("")].drop_duplicates(subset=["Index"], keep="first")

# 4) 可选：按 Index 数值排序（如果 Index 不是纯数字，就按字符串排序）
try:
    tmp["_Index_num"] = pd.to_numeric(tmp["Index"], errors="raise")
    tmp = tmp.sort_values("_Index_num").drop(columns=["_Index_num"])
except Exception:
    tmp = tmp.sort_values("Index")

tmp = tmp.reset_index(drop=True)

# 5) 输出你需要的两个列表
class_indices = tmp["Index"].tolist()
class_labels_fr = tmp["timel_label"].tolist()

print("num_classes =", len(class_labels_fr))
print("first 5 indices:", class_indices[:5])
print("first 5 labels:", class_labels_fr[:5])

# 如果你想在 notebook 里快速查看
display(tmp.head(10))


num_classes = 2333
first 5 indices: ['1', '2', '3', '4', '5']
first 5 labels: ['Joseph interprétant le songe du panetier', 'hérisson', 'château', 'moustique', 'juin']


,Index,timel_label
0,1,Joseph interprétant le songe du panetier
1,2,hérisson
2,3,château
3,4,moustique
4,5,juin
5,6,anatomie
6,7,tables de la loi
7,8,balance
8,9,mortier
9,10,infirmité


In [4]:
len(class_labels_fr), class_labels_fr[:5]


(2333,
 ['Joseph interprétant le songe du panetier',
  'hérisson',
  'château',
  'moustique',
  'juin'])

In [5]:
# 直接指定一张真实存在的图片
test_img = r"D:\O.D.I.L\Image\OneDrive_1_2026-1-6\000006.jpg"

import os
assert os.path.exists(test_img), f"图片不存在: {test_img}"

print("Using test_img:", test_img)


Using test_img: D:\O.D.I.L\Image\OneDrive_1_2026-1-6\000006.jpg


In [6]:
import torch
from PIL import Image

@torch.no_grad()
def score_image_label(image_path: str, label_text: str) -> float:
    """
    返回分数：越大越好。
    用 Qwen3-VL 的 chat template 正确插入 image 占位，从而避免 tokens=0 的报错。
    """
    image = Image.open(image_path).convert("RGB")

    # 关键：messages 的 content 里显式包含 {"type":"image"}，这样 chat template 会插入 image token
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": (
                        "Tu es un classificateur. "
                        "Choisis l'étiquette correcte pour cette image.\n"
                        "Réponds uniquement par l'étiquette exacte."
                    ),
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": label_text}
            ],
        },
    ]

    # 用 chat template 生成带 image token 的文本
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    # processor 同时接收 text + images
    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    # 放到模型设备
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # labels 必须与 input_ids 对齐长度；最稳做法：直接用 input_ids 作为 labels，
    # 并把“用户部分”mask 掉，只让 assistant 的那段参与 loss。
    # 下面用简单方式：labels = input_ids.clone()，再把非 assistant 部分置 -100。
    input_ids = inputs["input_ids"]
    labels = input_ids.clone()

    # 关键：找到 assistant 开始位置（不同模板略有差异；这里用一个通用办法：mask 掉全部，再只 unmask 最后 label_text 的 token）
    # 简化实现：只让最后 N 个 token（label_text 对应 token）参与 loss
    target_ids = processor.tokenizer(label_text, add_special_tokens=False).input_ids
    n = len(target_ids)
    labels[:] = -100
    labels[0, -n:] = input_ids[0, -n:]

    outputs = model(**inputs, labels=labels)
    loss = outputs.loss.item()
    return -loss


In [38]:
score = score_image_label(
    test_img,
    class_labels_fr[0]
)
print("score =", score)


[OK] label='Joseph interprétant le songe d...' loss=3.1495
score = -3.1495132446289062


In [39]:
labels_subset = class_labels_fr[:5]

scores, probs = predict_proba_for_image(
    image_path=test_img,
    labels=labels_subset,
    temperature=1.0
)

for lb, s, p in zip(labels_subset, scores, probs):
    print(f"{lb[:40]:40s}  score={float(s):7.3f}  proba={float(p):.3f}")


[OK] label='Joseph interprétant le songe d...' loss=3.1495
[OK] label='hérisson...' loss=0.0240
[OK] label='château...' loss=1.1178
[OK] label='moustique...' loss=1.3207
[OK] label='juin...' loss=0.7636
Joseph interprétant le songe du panetier  score= -3.150  proba=0.021
hérisson                                  score= -0.024  proba=0.470
château                                   score= -1.118  proba=0.157
moustique                                 score= -1.321  proba=0.128
juin                                      score= -0.764  proba=0.224


In [42]:
import numpy as np

# 你要测试的 class Index（按你给的）
target_indices = ["1370", "64", "2299", "2097", "915"]  # 用字符串最稳

# 防呆：确保 class_indices / class_labels_fr 存在
if "class_indices" not in globals() or "class_labels_fr" not in globals():
    raise NameError("class_indices 或 class_labels_fr 尚未定义。请先从 classes.tsv 生成它们。")

# 建立 Index -> label 的映射（保证能用 '1370' 这种键）
idx2label = {str(i): lb for i, lb in zip(class_indices, class_labels_fr)}

# 取出这 5 个 index 对应的法语标签
labels_subset = []
missing = []
for idx in target_indices:
    if str(idx) in idx2label:
        labels_subset.append(idx2label[str(idx)])
    else:
        missing.append(idx)

if missing:
    raise ValueError(f"这些 Index 在 classes.tsv 中找不到对应 label: {missing}")

print("Selected labels:")
for idx, lb in zip(target_indices, labels_subset):
    print(f"  {idx}: {lb}")

# 计算分数与 proba
scores, probs = predict_proba_for_image(
    image_path=test_img,
    labels=labels_subset,
    temperature=1.0
)

# 输出每个候选的分数与概率
print("\nScores / Probas:")
for idx, lb, s, p in zip(target_indices, labels_subset, scores, probs):
    print(f"Index={idx:>5s} | {lb[:60]:60s} | score={float(s):7.3f} | proba={float(p):.3f}")

# 给出 Top-1
best = int(np.argmax(probs))
print("\nTop-1:")
print(f"Index={target_indices[best]} | {labels_subset[best]} | proba={float(probs[best]):.3f}")


Selected labels:
  1370: cheval ailé
  64: miracle
  2299: serrure
  2097: guérison
  915: soldat
[OK] label='cheval ailé...' loss=3.4442
[OK] label='miracle...' loss=0.8802
[OK] label='serrure...' loss=0.3990
[OK] label='guérison...' loss=0.2648
[OK] label='soldat...' loss=0.8067

Scores / Probas:
Index= 1370 | cheval ailé                                                  | score= -3.444 | proba=0.014
Index=   64 | miracle                                                      | score= -0.880 | proba=0.178
Index= 2299 | serrure                                                      | score= -0.399 | proba=0.288
Index= 2097 | guérison                                                     | score= -0.265 | proba=0.329
Index=  915 | soldat                                                       | score= -0.807 | proba=0.191

Top-1:
Index=2097 | guérison | proba=0.329


In [43]:
# 换一张图片（请改成你真实存在的路径）
test_img = r"D:\O.D.I.L\Image\OneDrive_1_2026-1-6\000014.jpg"

import os
assert os.path.exists(test_img), f"图片不存在: {test_img}"

print("Using test_img:", test_img)


Using test_img: D:\O.D.I.L\Image\OneDrive_1_2026-1-6\000014.jpg


In [7]:
import numpy as np

# 你要测试的 class Index（按你给的）
target_indices = ["2000", "1051", "915", "168", "1656"]  # 用字符串最稳

# 防呆：确保 class_indices / class_labels_fr 存在
if "class_indices" not in globals() or "class_labels_fr" not in globals():
    raise NameError("class_indices 或 class_labels_fr 尚未定义。请先从 classes.tsv 生成它们。")

# 建立 Index -> label 的映射（保证能用 '1370' 这种键）
idx2label = {str(i): lb for i, lb in zip(class_indices, class_labels_fr)}

# 取出这 5 个 index 对应的法语标签
labels_subset = []
missing = []
for idx in target_indices:
    if str(idx) in idx2label:
        labels_subset.append(idx2label[str(idx)])
    else:
        missing.append(idx)

if missing:
    raise ValueError(f"这些 Index 在 classes.tsv 中找不到对应 label: {missing}")

print("Selected labels:")
for idx, lb in zip(target_indices, labels_subset):
    print(f"  {idx}: {lb}")


labels_subset = class_labels_fr[:5]

scores, probs = predict_proba_for_image(
    image_path=test_img,
    labels=labels_subset,
    temperature=1.0
)

for lb, s, p in zip(labels_subset, scores, probs):
    print(f"{lb[:40]:40s}  score={float(s):7.3f}  proba={float(p):.3f}")





# 计算分数与 proba
scores, probs = predict_proba_for_image(
    image_path=test_img,
    labels=labels_subset,
    temperature=1.0
)

# 输出每个候选的分数与概率
print("\nScores / Probas:")
for idx, lb, s, p in zip(target_indices, labels_subset, scores, probs):
    print(f"Index={idx:>5s} | {lb[:60]:60s} | score={float(s):7.3f} | proba={float(p):.3f}")

# 给出 Top-1
best = int(np.argmax(probs))
print("\nTop-1:")
print(f"Index={target_indices[best]} | {labels_subset[best]} | proba={float(probs[best]):.3f}")


Selected labels:
  2000: cheval
  1051: épée
  915: soldat
  168: aigle
  1656: pauvre


NameError: name 'predict_proba_for_image' is not defined

In [45]:
---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
Cell In[44], line 30
     27     print(f"  {idx}: {lb}")
     29 # 计算分数与 proba
---> 30 scores, probs = predict_proba_for_image(
     31     image_path=test_img,
     32     labels=labels_subset,
     33     temperature=1.0
     34 )
     36 # 输出每个候选的分数与概率
     37 print("\nScores / Probas:")

Cell In[11], line 4, in predict_proba_for_image(image_path, labels, temperature)
      3 def predict_proba_for_image(image_path: str, labels: list[str], temperature: float = 1.0):
----> 4     scores = np.array([score_image_label(image_path, lb) for lb in labels], dtype=np.float32)
      5     scores = scores / float(temperature)
      7     # softmax

Cell In[11], line 4, in <listcomp>(.0)
      3 def predict_proba_for_image(image_path: str, labels: list[str], temperature: float = 1.0):
----> 4     scores = np.array([score_image_label(image_path, lb) for lb in labels], dtype=np.float32)
      5     scores = scores / float(temperature)
      7     # softmax

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\utils\_contextlib.py:116, in context_decorator.<locals>.decorate_context(*args, **kwargs)
    113 @functools.wraps(func)
    114 def decorate_context(*args, **kwargs):
    115     with ctx_factory():
--> 116         return func(*args, **kwargs)

Cell In[37], line 39, in score_image_label(image_path, label_text)
     36 labels[:] = -100
     37 labels[0, -n:] = input_ids[0, -n:]
---> 39 outputs = model(**inputs, labels=labels)
     40 loss = outputs.loss.item()
     42 print(f"[OK] label='{label_text[:30]}...' loss={loss:.4f}")

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1736, in Module._wrapped_call_impl(self, *args, **kwargs)
   1734     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
   1735 else:
-> 1736     return self._call_impl(*args, **kwargs)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1747, in Module._call_impl(self, *args, **kwargs)
   1742 # If we don't have any hooks, we want to skip the rest of the logic in
   1743 # this function, and just call forward.
   1744 if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
   1745         or _global_backward_pre_hooks or _global_backward_hooks
   1746         or _global_forward_hooks or _global_forward_pre_hooks):
-> 1747     return forward_call(*args, **kwargs)
   1749 result = None
   1750 called_always_called_hooks = set()

File ~\miniconda3\envs\pairgpu\Lib\site-packages\accelerate\hooks.py:175, in add_hook_to_module.<locals>.new_forward(module, *args, **kwargs)
    173         output = module._old_forward(*args, **kwargs)
    174 else:
--> 175     output = module._old_forward(*args, **kwargs)
    176 return module._hf_hook.post_forward(module, output)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\utils\generic.py:1072, in check_model_inputs.<locals>.wrapped_fn.<locals>.wrapper(self, *args, **kwargs)
   1069                 monkey_patched_layers.append((module, original_forward))
   1071 try:
-> 1072     outputs = func(self, *args, **kwargs)
   1073 except TypeError as original_exception:
   1074     # If we get a TypeError, it's possible that the model is not receiving the recordable kwargs correctly.
   1075     # Get a TypeError even after removing the recordable kwargs -> re-raise the original exception
   1076     # Otherwise -> we're probably missing `**kwargs` in the decorated function
   1077     kwargs_without_recordable = {k: v for k, v in kwargs.items() if k not in recordable_keys}

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\models\qwen3_vl\modeling_qwen3_vl.py:1344, in Qwen3VLForConditionalGeneration.forward(self, input_ids, attention_mask, position_ids, past_key_values, inputs_embeds, labels, pixel_values, pixel_values_videos, image_grid_thw, video_grid_thw, cache_position, logits_to_keep, **kwargs)
   1314 @check_model_inputs()
   1315 def forward(
   1316     self,
   (...)   1329     **kwargs: Unpack[TransformersKwargs],
   1330 ) -> Union[tuple, Qwen3VLCausalLMOutputWithPast]:
   1331     r"""
   1332     labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
   1333         Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,
   (...)   1342         TODO: Add example
   1343     """
-> 1344     outputs = self.model(
   1345         input_ids=input_ids,
   1346         pixel_values=pixel_values,
   1347         pixel_values_videos=pixel_values_videos,
   1348         image_grid_thw=image_grid_thw,
   1349         video_grid_thw=video_grid_thw,
   1350         position_ids=position_ids,
   1351         attention_mask=attention_mask,
   1352         past_key_values=past_key_values,
   1353         inputs_embeds=inputs_embeds,
   1354         cache_position=cache_position,
   1355         **kwargs,
   1356     )
   1358     hidden_states = outputs[0]
   1360     # Only compute necessary logits, and do not upcast them to float if we are not computing the loss

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1736, in Module._wrapped_call_impl(self, *args, **kwargs)
   1734     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
   1735 else:
-> 1736     return self._call_impl(*args, **kwargs)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1747, in Module._call_impl(self, *args, **kwargs)
   1742 # If we don't have any hooks, we want to skip the rest of the logic in
   1743 # this function, and just call forward.
   1744 if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
   1745         or _global_backward_pre_hooks or _global_backward_hooks
   1746         or _global_forward_hooks or _global_forward_pre_hooks):
-> 1747     return forward_call(*args, **kwargs)
   1749 result = None
   1750 called_always_called_hooks = set()

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\utils\generic.py:1072, in check_model_inputs.<locals>.wrapped_fn.<locals>.wrapper(self, *args, **kwargs)
   1069                 monkey_patched_layers.append((module, original_forward))
   1071 try:
-> 1072     outputs = func(self, *args, **kwargs)
   1073 except TypeError as original_exception:
   1074     # If we get a TypeError, it's possible that the model is not receiving the recordable kwargs correctly.
   1075     # Get a TypeError even after removing the recordable kwargs -> re-raise the original exception
   1076     # Otherwise -> we're probably missing `**kwargs` in the decorated function
   1077     kwargs_without_recordable = {k: v for k, v in kwargs.items() if k not in recordable_keys}

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\models\qwen3_vl\modeling_qwen3_vl.py:1223, in Qwen3VLModel.forward(self, input_ids, attention_mask, position_ids, past_key_values, inputs_embeds, pixel_values, pixel_values_videos, image_grid_thw, video_grid_thw, cache_position, **kwargs)
   1220         position_ids = position_ids.add(delta)
   1221         position_ids = position_ids.unsqueeze(0).expand(3, -1, -1)
-> 1223 outputs = self.language_model(
   1224     input_ids=None,
   1225     position_ids=position_ids,
   1226     attention_mask=attention_mask,
   1227     past_key_values=past_key_values,
   1228     inputs_embeds=inputs_embeds,
   1229     cache_position=cache_position,
   1230     visual_pos_masks=visual_pos_masks,
   1231     deepstack_visual_embeds=deepstack_visual_embeds,
   1232     **kwargs,
   1233 )
   1235 return Qwen3VLModelOutputWithPast(
   1236     last_hidden_state=outputs.last_hidden_state,
   1237     past_key_values=outputs.past_key_values,
   1238     rope_deltas=self.rope_deltas,
   1239 )

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1736, in Module._wrapped_call_impl(self, *args, **kwargs)
   1734     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
   1735 else:
-> 1736     return self._call_impl(*args, **kwargs)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1747, in Module._call_impl(self, *args, **kwargs)
   1742 # If we don't have any hooks, we want to skip the rest of the logic in
   1743 # this function, and just call forward.
   1744 if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
   1745         or _global_backward_pre_hooks or _global_backward_hooks
   1746         or _global_forward_hooks or _global_forward_pre_hooks):
-> 1747     return forward_call(*args, **kwargs)
   1749 result = None
   1750 called_always_called_hooks = set()

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\utils\generic.py:1072, in check_model_inputs.<locals>.wrapped_fn.<locals>.wrapper(self, *args, **kwargs)
   1069                 monkey_patched_layers.append((module, original_forward))
   1071 try:
-> 1072     outputs = func(self, *args, **kwargs)
   1073 except TypeError as original_exception:
   1074     # If we get a TypeError, it's possible that the model is not receiving the recordable kwargs correctly.
   1075     # Get a TypeError even after removing the recordable kwargs -> re-raise the original exception
   1076     # Otherwise -> we're probably missing `**kwargs` in the decorated function
   1077     kwargs_without_recordable = {k: v for k, v in kwargs.items() if k not in recordable_keys}

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\models\qwen3_vl\modeling_qwen3_vl.py:850, in Qwen3VLTextModel.forward(self, input_ids, attention_mask, position_ids, past_key_values, inputs_embeds, use_cache, cache_position, visual_pos_masks, deepstack_visual_embeds, **kwargs)
    848 # decoder layers
    849 for layer_idx, decoder_layer in enumerate(self.layers):
--> 850     layer_outputs = decoder_layer(
    851         hidden_states,
    852         attention_mask=attention_mask,
    853         position_ids=text_position_ids,
    854         past_key_values=past_key_values,
    855         cache_position=cache_position,
    856         position_embeddings=position_embeddings,
    857         **kwargs,
    858     )
    859     hidden_states = layer_outputs
    861     # add visual features to the hidden states of first several layers

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\modeling_layers.py:94, in GradientCheckpointingLayer.__call__(self, *args, **kwargs)
     91         logger.warning_once(message)
     93     return self._gradient_checkpointing_func(partial(super().__call__, **kwargs), *args)
---> 94 return super().__call__(*args, **kwargs)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1736, in Module._wrapped_call_impl(self, *args, **kwargs)
   1734     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
   1735 else:
-> 1736     return self._call_impl(*args, **kwargs)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1747, in Module._call_impl(self, *args, **kwargs)
   1742 # If we don't have any hooks, we want to skip the rest of the logic in
   1743 # this function, and just call forward.
   1744 if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
   1745         or _global_backward_pre_hooks or _global_backward_hooks
   1746         or _global_forward_hooks or _global_forward_pre_hooks):
-> 1747     return forward_call(*args, **kwargs)
   1749 result = None
   1750 called_always_called_hooks = set()

File ~\miniconda3\envs\pairgpu\Lib\site-packages\accelerate\hooks.py:175, in add_hook_to_module.<locals>.new_forward(module, *args, **kwargs)
    173         output = module._old_forward(*args, **kwargs)
    174 else:
--> 175     output = module._old_forward(*args, **kwargs)
    176 return module._hf_hook.post_forward(module, output)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\utils\deprecation.py:172, in deprecate_kwarg.<locals>.wrapper.<locals>.wrapped_func(*args, **kwargs)
    168 elif minimum_action in (Action.NOTIFY, Action.NOTIFY_ALWAYS) and not is_torchdynamo_compiling():
    169     # DeprecationWarning is ignored by default, so we use FutureWarning instead
    170     warnings.warn(message, FutureWarning, stacklevel=2)
--> 172 return func(*args, **kwargs)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\models\qwen3_vl\modeling_qwen3_vl.py:516, in Qwen3VLTextDecoderLayer.forward(self, hidden_states, position_embeddings, attention_mask, position_ids, past_key_values, use_cache, cache_position, **kwargs)
    514 # Fully Connected
    515 residual = hidden_states
--> 516 hidden_states = self.post_attention_layernorm(hidden_states)
    517 hidden_states = self.mlp(hidden_states)
    518 hidden_states = residual + hidden_states

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1736, in Module._wrapped_call_impl(self, *args, **kwargs)
   1734     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
   1735 else:
-> 1736     return self._call_impl(*args, **kwargs)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\torch\nn\modules\module.py:1747, in Module._call_impl(self, *args, **kwargs)
   1742 # If we don't have any hooks, we want to skip the rest of the logic in
   1743 # this function, and just call forward.
   1744 if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
   1745         or _global_backward_pre_hooks or _global_backward_hooks
   1746         or _global_forward_hooks or _global_forward_pre_hooks):
-> 1747     return forward_call(*args, **kwargs)
   1749 result = None
   1750 called_always_called_hooks = set()

File ~\miniconda3\envs\pairgpu\Lib\site-packages\accelerate\hooks.py:175, in add_hook_to_module.<locals>.new_forward(module, *args, **kwargs)
    173         output = module._old_forward(*args, **kwargs)
    174 else:
--> 175     output = module._old_forward(*args, **kwargs)
    176 return module._hf_hook.post_forward(module, output)

File ~\miniconda3\envs\pairgpu\Lib\site-packages\transformers\models\qwen3_vl\modeling_qwen3_vl.py:352, in Qwen3VLTextRMSNorm.forward(self, hidden_states)
    350 variance = hidden_states.pow(2).mean(-1, keepdim=True)
    351 hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
--> 352 return self.weight * hidden_states.to(input_dtype)

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

SyntaxError: invalid syntax (1792379648.py, line 1)